In [0]:
%sql
USE CATALOG ws_banking_etl2;
CREATE SCHEMA IF NOT EXISTS gold;


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

transactions = spark.read.table("silver.transactions")


1. Which day(s) of the week sees the highest number of fraudulent transactions?

In [0]:
fraud_by_weekday = (
    transactions
        .filter(col("is_fraud") == True)
        .groupBy(date_format(col("transaction_date"), "EEEE").alias("day_of_week"))
        .count()
        .withColumnRenamed("count", "fraud_transaction_count")
        .orderBy(desc("fraud_transaction_count"))
)
fraud_by_weekday.show()

+-----------+-----------------------+
|day_of_week|fraud_transaction_count|
+-----------+-----------------------+
|     Sunday|                   2646|
|     Friday|                   2284|
|   Thursday|                   2082|
|    Tuesday|                   2037|
|     Monday|                   1747|
|   Saturday|                   1434|
|  Wednesday|                   1102|
+-----------+-----------------------+



In [0]:
fraud_by_weekday.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.fraud_by_weekday")


2. What is the trend of the fraud rate (fraudulent transactions divided by total) over the past month?

Fraud rate = fraud_transactions / total_transactions

In [0]:
from pyspark.sql.functions import current_date, date_sub

# if the date is not up todate, we can use the following to get the last month's data
# Get latest date in dataset
max_date = transactions.select(max("transaction_date")).collect()[0][0]

last_month_data = transactions.filter(
    col("transaction_date") >= date_sub(lit(max_date), 30)
)
# last_month_data = transactions.filter(
#     col("transaction_date") >= date_sub(current_date(), 30)
# )

daily_stats = (
    last_month_data
        .groupBy("transaction_date")
        .agg(
            count("*").alias("total_transactions"),
            sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions")
        )
        .withColumn(
            "fraud_rate",
            col("fraud_transactions") / col("total_transactions")
        )
        .orderBy("transaction_date")
)
daily_stats.show()

+----------------+------------------+------------------+--------------------+
|transaction_date|total_transactions|fraud_transactions|          fraud_rate|
+----------------+------------------+------------------+--------------------+
|      2019-10-01|              3363|                 8|0.002378828426999...|
|      2019-10-02|              3541|                 0|                 0.0|
|      2019-10-03|              3897|                10|0.002566076469078...|
|      2019-10-04|              3995|                11|0.002753441802252816|
|      2019-10-05|              3862|                13|0.003366131538063...|
|      2019-10-06|              3973|                 0|                 0.0|
|      2019-10-07|              3906|                 0|                 0.0|
|      2019-10-08|              3478|                20|0.005750431282346176|
|      2019-10-09|              3585|                 0|                 0.0|
|      2019-10-10|              3887|                10|0.002572

In [0]:
daily_stats.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.fraud_rate_trend")


3. Which users have the largest number of flagged (“is_fraud = true”) transactions?

In [0]:
top_fraud_users = (
    transactions
        .filter(col("is_fraud") == True)
        .groupBy("client_id")
        .count()
        .withColumnRenamed("count", "fraud_transaction_count")
        .orderBy(desc("fraud_transaction_count"))
)
top_fraud_users.show()

+---------+-----------------------+
|client_id|fraud_transaction_count|
+---------+-----------------------+
|     1102|                     58|
|      209|                     52|
|       27|                     45|
|      155|                     44|
|     1128|                     43|
|     1741|                     42|
|     1851|                     42|
|      989|                     42|
|     1649|                     41|
|      692|                     39|
|     1926|                     39|
|     1416|                     39|
|      359|                     39|
|     1571|                     39|
|      408|                     39|
|     1992|                     38|
|     1725|                     38|
|     1569|                     37|
|     1341|                     37|
|      615|                     36|
+---------+-----------------------+
only showing top 20 rows


In [0]:
top_fraud_users.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.top_fraud_users")


4. Are there any users showing a sharp rise in transaction amount compared to their weekly average?

In [0]:
weekly_avg = (
    transactions
        .withColumn("week", weekofyear("transaction_date"))
        .groupBy("client_id", "week")
        .agg(avg("amount").alias("weekly_avg_amount"))
)
weekly_avg.show()

+---------+----+-----------------+
|client_id|week|weekly_avg_amount|
+---------+----+-----------------+
|     1075|  53|      24.56346939|
|     1697|  53|      22.13103448|
|     1096|   2|      48.50288630|
|      938|   3|      47.59157360|
|     1303|   3|      80.66028169|
|     1820|   3|      53.99805195|
|     1274|   5|      57.78630225|
|       62|   5|      25.92661417|
|      172|   6|      47.41225490|
|      363|   7|      51.10445026|
|      882|   8|      33.66490323|
|      637|   9|      50.97441088|
|      369|   9|      67.93521912|
|      524|   9|      42.51919192|
|      452|  10|      53.36190000|
|     1529|  10|      30.94982609|
|     1669|  10|      64.70087302|
|      876|  11|      46.76733333|
|       40|  12|      38.40398438|
|      573|  12|      20.48025316|
+---------+----+-----------------+
only showing top 20 rows


In [0]:
daily_total = (
    transactions
        .groupBy("client_id", "transaction_date")
        .agg(sum("amount").alias("daily_total_amount"))
)
daily_total.show()

+---------+----------------+------------------+
|client_id|transaction_date|daily_total_amount|
+---------+----------------+------------------+
|     1179|      2010-01-01|           49.5100|
|      462|      2010-01-01|           89.3800|
|     1121|      2010-01-02|          266.5300|
|     1512|      2010-01-02|          272.2600|
|      230|      2010-01-02|           23.2500|
|      752|      2010-01-03|          167.9600|
|      571|      2010-01-03|           82.8200|
|      580|      2010-01-04|          397.2200|
|      865|      2010-01-04|          294.3000|
|      457|      2010-01-04|          389.2800|
|     1177|      2010-01-04|          113.3600|
|      335|      2010-01-05|          106.2100|
|      772|      2010-01-05|            1.7300|
|     1595|      2010-01-05|          301.0100|
|     1826|      2010-01-06|            8.5700|
|     1378|      2010-01-06|          173.9500|
|      250|      2010-01-07|          150.7300|
|     1619|      2010-01-07|           5

In [0]:
spike_users = (
    daily_total
        .withColumn("week", weekofyear("transaction_date"))
        .join(weekly_avg, ["client_id", "week"], "inner")
        .withColumn(
            "is_spike",
            col("daily_total_amount") > col("weekly_avg_amount") * 2
        )
        .filter(col("is_spike") == True)
        .orderBy(desc("daily_total_amount"))
)
spike_users.show()

+---------+----+----------------+------------------+-----------------+--------+
|client_id|week|transaction_date|daily_total_amount|weekly_avg_amount|is_spike|
+---------+----+----------------+------------------+-----------------+--------+
|      708|  38|      2010-09-22|         6820.2000|     167.99141935|    true|
|     1081|   4|      2019-01-27|         6723.4000|      61.41163889|    true|
|     1164|  22|      2017-05-30|         6646.7500|     139.18192982|    true|
|     1699|  35|      2010-09-05|         6341.2500|      81.22237179|    true|
|     1259|  15|      2012-04-10|         5951.6200|     113.11217195|    true|
|      708|   2|      2019-01-13|         5911.0200|     148.16597403|    true|
|     1487|  21|      2013-05-22|         5905.3900|     107.39043478|    true|
|     1156|  19|      2018-05-09|         5898.9600|      94.24698113|    true|
|      278|  43|      2014-10-24|         5811.3400|      86.83112281|    true|
|     1983|  49|      2010-12-11|       

In [0]:
spike_users.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.users_spike_vs_weekly_avg")


5. Which merchant categories exhibit the highest fraud rate?

In [0]:
fraud_rate_by_mcc = (
    transactions
        .groupBy("mcc", "mcc_description")
        .agg(
            count("*").alias("total_transactions"),
            sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions")
        )
        .withColumn(
            "fraud_rate",
            col("fraud_transactions") / col("total_transactions")
        )
        .orderBy(desc("fraud_rate"))
)
fraud_rate_by_mcc.show()

+----+--------------------+------------------+------------------+--------------------+
| mcc|     mcc_description|total_transactions|fraud_transactions|          fraud_rate|
+----+--------------------+------------------+------------------+--------------------+
|4411|        Cruise Lines|               428|               165|  0.3855140186915888|
|5733|Music Stores - Mu...|               319|                76| 0.23824451410658307|
|3006|Miscellaneous Fab...|               351|                29| 0.08262108262108261|
|5045|Computers, Comput...|              2793|               204| 0.07303974221267455|
|3144|Floor Covering St...|               334|                23|  0.0688622754491018|
|5732|  Electronics Stores|              6997|               402|  0.0574531942260969|
|3005|Miscellaneous Met...|               391|                22|0.056265984654731455|
|3009|Fabricated Struct...|               408|                22| 0.05392156862745098|
|5094|Precious Stones a...|              51

In [0]:
fraud_rate_by_mcc.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.fraud_rate_by_mcc")


6. Are there specific merchants with unusually high fraud volume?

In [0]:
high_fraud_merchants = (
    transactions
        .filter(col("is_fraud") == True)
        .groupBy("merchant_id")
        .count()
        .withColumnRenamed("count", "fraud_transaction_count")
        .orderBy(desc("fraud_transaction_count"))
)
high_fraud_merchants.show()

+-----------+-----------------------+
|merchant_id|fraud_transaction_count|
+-----------+-----------------------+
|      60569|                    764|
|      27092|                    725|
|      48919|                    332|
|      99370|                    302|
|      32858|                    289|
|      76639|                    286|
|      83018|                    236|
|      67570|                    225|
|      31893|                    214|
|      34490|                    197|
|      47399|                    176|
|      59474|                    151|
|      18215|                    150|
|      88260|                    149|
|      41375|                    147|
|      39261|                    139|
|      52923|                    137|
|      85247|                    134|
|      46284|                    131|
|      49637|                    129|
+-----------+-----------------------+
only showing top 20 rows


In [0]:
high_fraud_merchants.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.merchants_high_fraud_volume")


7. What’s the average transaction amount for fraud vs non-fraud transactions?

In [0]:
avg_amount = (
    transactions
        .groupBy("is_fraud")
        .agg(
            avg("amount").alias("avg_transaction_amount")
        )
)
avg_amount.show()

+--------+----------------------+
|is_fraud|avg_transaction_amount|
+--------+----------------------+
|   false|           42.90858094|
|    true|          110.23468197|
+--------+----------------------+



In [0]:
avg_amount.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.avg_amount_fraud_vs_nonfraud")


8. Which merchant category has the highest total fraud amount?

In [0]:
mcc_fraud_amount = (
    transactions
        .filter(col("is_fraud") == True)
        .groupBy("mcc", "mcc_description")
        .agg(
            sum("amount").alias("total_fraud_amount")
        )
        .orderBy(desc("total_fraud_amount"))
)
mcc_fraud_amount.show()

+----+--------------------+------------------+
| mcc|     mcc_description|total_fraud_amount|
+----+--------------------+------------------+
|5311|   Department Stores|       225647.1900|
|4411|        Cruise Lines|       185946.7800|
|5300|     Wholesale Clubs|       113827.6500|
|5310|     Discount Stores|        81214.8900|
|4829|      Money Transfer|        66101.5200|
|5732|  Electronics Stores|        61171.3800|
|5712|Furniture, Home F...|        56989.4500|
|5719|Miscellaneous Hom...|        34238.4500|
|4814|Telecommunication...|        33625.0400|
|5651|Family Clothing S...|        30051.5600|
|5912|Drug Stores and P...|        29926.5900|
|5094|Precious Stones a...|        26575.9600|
|5045|Computers, Comput...|        25615.1100|
|5815|Digital Goods - M...|        24223.9600|
|3722|  Passenger Railways|        19976.4400|
|3504|  Gardening Supplies|        19593.0300|
|5211|Lumber and Buildi...|        18776.3600|
|5411|Grocery Stores, S...|        18095.6800|
|5816|Digital

In [0]:
mcc_fraud_amount.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.mcc_highest_total_fraud_amount")


9. What are the total monetary losses due to fraud each day?

In [0]:
daily_fraud_losses = (
    transactions
        .filter(col("is_fraud") == True)
        .groupBy("transaction_date")
        .agg(
            sum("amount").alias("total_fraud_loss")
        )
        .orderBy("transaction_date")
)
daily_fraud_losses.show()

+----------------+----------------+
|transaction_date|total_fraud_loss|
+----------------+----------------+
|      2010-01-01|          0.1900|
|      2010-01-03|        339.0000|
|      2010-01-04|         11.6400|
|      2010-01-05|          8.7600|
|      2010-01-07|        -48.4600|
|      2010-01-08|        383.2400|
|      2010-01-09|         23.1000|
|      2010-01-10|        530.0100|
|      2010-01-11|        302.7000|
|      2010-01-12|       1014.4100|
|      2010-01-13|        -95.7000|
|      2010-01-15|         24.7100|
|      2010-01-16|       1294.0100|
|      2010-01-17|        262.0200|
|      2010-01-18|        485.2800|
|      2010-01-19|         33.2400|
|      2010-01-22|        922.0600|
|      2010-01-23|         11.1400|
|      2010-01-24|        154.7800|
|      2010-01-25|        396.2100|
+----------------+----------------+
only showing top 20 rows


In [0]:
daily_fraud_losses.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.daily_fraud_losses")


10. How many unique users commit fraudulent transactions per week?

In [0]:
weekly_unique_fraud_users = (
    transactions
        .filter(col("is_fraud") == True)
        .withColumn("year", year("transaction_date"))
        .withColumn("week", weekofyear("transaction_date"))
        .groupBy("year", "week")
        .agg(
            countDistinct("client_id").alias("unique_fraud_users"),
            count("*").alias("fraud_transaction_count")
        )
        .orderBy("year", "week")
)
weekly_unique_fraud_users.show()

+----+----+------------------+-----------------------+
|year|week|unique_fraud_users|fraud_transaction_count|
+----+----+------------------+-----------------------+
|2010|   1|                 5|                     15|
|2010|   2|                 5|                     11|
|2010|   3|                 7|                     18|
|2010|   4|                17|                     61|
|2010|   5|                16|                     70|
|2010|   6|                12|                     70|
|2010|   7|                13|                     57|
|2010|   8|                13|                     62|
|2010|   9|                10|                     65|
|2010|  10|                 6|                     29|
|2010|  11|                14|                     75|
|2010|  12|                11|                     77|
|2010|  13|                11|                     48|
|2010|  14|                13|                     70|
|2010|  15|                11|                     51|
|2010|  16

In [0]:
weekly_unique_fraud_users.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.weekly_unique_fraud_users")


11. Do fraud patterns show seasonal or monthly spikes?

In [0]:
monthly_fraud_trends = (
    transactions
        .withColumn("year", year("transaction_date"))
        .withColumn("month", month("transaction_date"))
        .groupBy("year", "month")
        .agg(
            count("*").alias("total_transactions"),
            sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions"),
            sum(when(col("is_fraud") == True, col("amount")).otherwise(0)).alias("total_fraud_amount")
        )
        .withColumn(
            "fraud_rate",
            col("fraud_transactions") / col("total_transactions")
        )
        .orderBy("year", "month")
)
monthly_fraud_trends.show()

+----+-----+------------------+------------------+------------------+--------------------+
|year|month|total_transactions|fraud_transactions|total_fraud_amount|          fraud_rate|
+----+-----+------------------+------------------+------------------+--------------------+
|2010|    1|            101209|               107|        12253.5700|0.001057218231580...|
|2010|    2|             93470|               259|        32488.1100|0.002770942548411255|
|2010|    3|            103345|               261|        37454.2000|0.002525521312109923|
|2010|    4|            100169|               237|        29130.4200|0.002366001457536...|
|2010|    5|            104773|               274|        27258.8600|0.002615177574375...|
|2010|    6|            102677|               182|        15926.2900|0.001772548866834...|
|2010|    7|            106034|               244|        28099.7700|0.002301148688156629|
|2010|    8|            107547|               229|        24878.7500|0.002129301607669205|

In [0]:
monthly_fraud_trends.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.monthly_fraud_trends")


12. How has user behavior changed before versus after a fraudulent event?

In [0]:
first_fraud = (
    transactions
        .filter(col("is_fraud") == True)
        .groupBy("client_id")
        .agg(min("transaction_date").alias("first_fraud_date"))
)


In [0]:
from pyspark.sql.functions import when

behavior_split = (
    transactions
        .join(first_fraud, "client_id", "inner")
        .withColumn(
            "period",
            when(col("transaction_date") < col("first_fraud_date"), "BEFORE_FRAUD")
            .otherwise("AFTER_FRAUD")
        )
)


In [0]:
user_behavior_change = (
    behavior_split
        .groupBy("period")
        .agg(
            avg("amount").alias("avg_transaction_amount"),
            count("*").alias("transaction_count"),
            sum("amount").alias("total_spending")
        )
)
user_behavior_change.show()

+------------+----------------------+-----------------+--------------+
|      period|avg_transaction_amount|transaction_count|total_spending|
+------------+----------------------+-----------------+--------------+
| AFTER_FRAUD|           42.92939435|          7763085|333264537.3300|
|BEFORE_FRAUD|           42.93583216|          5321437|228480325.8800|
+------------+----------------------+-----------------+--------------+



In [0]:
user_behavior_change.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.user_behavior_before_after_fraud")


13. Are fraudulent transactions more common on high-value purchases compared to low-value purchases?

In [0]:
value_segmented = (
    transactions
        .withColumn(
            "value_segment",
            when(col("amount") < 50, "Low")
            .when((col("amount") >= 50) & (col("amount") <= 200), "Medium")
            .otherwise("High")
        )
)


In [0]:
fraud_by_value = (
    value_segmented
        .groupBy("value_segment")
        .agg(
            count("*").alias("total_transactions"),
            sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions")
        )
        .withColumn(
            "fraud_rate",
            col("fraud_transactions") / col("total_transactions")
        )
        .orderBy(desc("fraud_rate"))
)
fraud_by_value.show()

+-------------+------------------+------------------+--------------------+
|value_segment|total_transactions|fraud_transactions|          fraud_rate|
+-------------+------------------+------------------+--------------------+
|         High|            319101|              2194|0.006875566043353045|
|       Medium|           4139987|              5707|0.001378506744103...|
|          Low|           8846827|              5431|6.138924158910308E-4|
+-------------+------------------+------------------+--------------------+



In [0]:
fraud_by_value.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.fraud_by_value_segment")
